In [2]:
# ============================================================
# C4 — DOMAIN-CONTEXT HYBRID
# Step 1: Import Libraries
# ============================================================

import pandas as pd
import numpy as np
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    explained_variance_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# ============================================================
# C4 — Step 2: Load Energy and Weather Datasets
# ============================================================

energy_path = "../../datasets/Energy_consumption.csv"
weather_path = "../../datasets/weather_history.csv"

energy_df = pd.read_csv(energy_path)
weather_df = pd.read_csv(weather_path)

print("Energy Dataset Shape:", energy_df.shape)
print("Weather Dataset Shape:", weather_df.shape)

print("\nEnergy Columns:")
print(energy_df.columns.tolist())

print("\nWeather Columns:")
print(weather_df.columns.tolist())

Energy Dataset Shape: (1000, 11)
Weather Dataset Shape: (96453, 12)

Energy Columns:
['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption']

Weather Columns:
['Formatted Date', 'Summary', 'Precip Type', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Loud Cover', 'Pressure (millibars)', 'Daily Summary']


In [4]:
import os

print("Current Working Directory:", os.getcwd())
print("Files in current directory:", os.listdir())

Current Working Directory: C:\Users\umash\OneDrive\Desktop\shankar\DAV\Project_Implementation\Hybrid_Model\Hybrid_Experiments
Files in current directory: ['.ipynb_checkpoints', 'C4_Domain_Context_Hybrid.ipynb.ipynb']


In [5]:
# ============================================================
# C4 — Step 3: Check Target and Potential Leakage
# ============================================================

target = "EnergyConsumption"

print("Energy Target:")
print(target)

print("\nEnergy Data Types:")
print(energy_df.dtypes)

print("\nChecking target statistics:")
print(energy_df[target].describe())

print("\nPotentially derived / leakage-prone features:")
print([
    "Renewable_Energy_Ratio",
    "EnergyConsumption"
])

print("\nWeather columns:")
print(weather_df.columns.tolist())

Energy Target:
EnergyConsumption

Energy Data Types:
Timestamp             object
Temperature          float64
Humidity             float64
SquareFootage        float64
Occupancy              int64
HVACUsage             object
LightingUsage         object
RenewableEnergy      float64
DayOfWeek             object
Holiday               object
EnergyConsumption    float64
dtype: object

Checking target statistics:
count    1000.000000
mean       77.055873
std         8.144112
min        53.263278
25%        71.544690
50%        76.943696
75%        82.921742
max        99.201120
Name: EnergyConsumption, dtype: float64

Potentially derived / leakage-prone features:
['Renewable_Energy_Ratio', 'EnergyConsumption']

Weather columns:
['Formatted Date', 'Summary', 'Precip Type', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Loud Cover', 'Pressure (millibars)', 'Daily Summary']


In [7]:
# ============================================================
# C4 — Step 4A: Load Feature-Engineered Energy Dataset
# ============================================================

energy_fe_path = "../../Feature_engineered_datasets/energy_feature_engineered.csv"

energy_df = pd.read_csv(energy_fe_path)

print("Feature-Engineered Energy Dataset Shape:", energy_df.shape)

print("\nEnergy Columns:")
print(energy_df.columns.tolist())

Feature-Engineered Energy Dataset Shape: (1000, 19)

Energy Columns:
['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status', 'Energy_Per_Occupant', 'Renewable_Energy_Ratio', 'Energy_Per_SquareFoot']


In [8]:
# ============================================================
# C4 — Step 4: Create Leakage-Free Energy Feature Set
# ============================================================

energy_features = [
    "Temperature",
    "Humidity",
    "SquareFootage",
    "Occupancy",
    "HVACUsage",
    "LightingUsage",
    "RenewableEnergy",
    "DayOfWeek",
    "Holiday",
    "Hour",
    "Month",
    "Weekend",
    "HVAC_Status",
    "Lighting_Status"
]

X_energy = energy_df[energy_features].copy()
y_energy = energy_df["EnergyConsumption"].copy()

print("Selected Energy Features:")
print(X_energy.columns.tolist())

print("\nFeature Matrix Shape:", X_energy.shape)
print("Target Shape:", y_energy.shape)

print("\nMissing Values:")
print(X_energy.isnull().sum())

print("\nTarget:")
print("EnergyConsumption")

print("\nExcluded Leakage-Prone Features:")
print([
    "Energy_Per_Occupant",
    "Renewable_Energy_Ratio",
    "Energy_Per_SquareFoot"
])

Selected Energy Features:
['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status']

Feature Matrix Shape: (1000, 14)
Target Shape: (1000,)

Missing Values:
Temperature        0
Humidity           0
SquareFootage      0
Occupancy          0
HVACUsage          0
LightingUsage      0
RenewableEnergy    0
DayOfWeek          0
Holiday            0
Hour               0
Month              0
Weekend            0
HVAC_Status        0
Lighting_Status    0
dtype: int64

Target:
EnergyConsumption

Excluded Leakage-Prone Features:
['Energy_Per_Occupant', 'Renewable_Energy_Ratio', 'Energy_Per_SquareFoot']


In [9]:
# ============================================================
# C4 — Step 5: Check Data Types
# ============================================================

print("Energy Feature Data Types:\n")
print(X_energy.dtypes)

print("\nCategorical Columns:")
print(X_energy.select_dtypes(include=["object"]).columns.tolist())

Energy Feature Data Types:

Temperature        float64
Humidity           float64
SquareFootage      float64
Occupancy            int64
HVACUsage           object
LightingUsage       object
RenewableEnergy    float64
DayOfWeek           object
Holiday             object
Hour                 int64
Month                int64
Weekend             object
HVAC_Status          int64
Lighting_Status      int64
dtype: object

Categorical Columns:
['HVACUsage', 'LightingUsage', 'DayOfWeek', 'Holiday', 'Weekend']


In [10]:
# ============================================================
# C4 — Step 6: Encode Categorical Features
# ============================================================

X_energy_encoded = pd.get_dummies(
    X_energy,
    columns=[
        "HVACUsage",
        "LightingUsage",
        "DayOfWeek",
        "Holiday",
        "Weekend"
    ],
    drop_first=True,
    dtype=int
)

print("Encoded Feature Matrix Shape:", X_energy_encoded.shape)

print("\nEncoded Columns:")
print(X_energy_encoded.columns.tolist())

print("\nData Types:")
print(X_energy_encoded.dtypes)

print("\nMissing Values:")
print(X_energy_encoded.isnull().sum().sum())

Encoded Feature Matrix Shape: (1000, 19)

Encoded Columns:
['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'RenewableEnergy', 'Hour', 'Month', 'HVAC_Status', 'Lighting_Status', 'HVACUsage_On', 'LightingUsage_On', 'DayOfWeek_Monday', 'DayOfWeek_Saturday', 'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday', 'DayOfWeek_Wednesday', 'Holiday_Yes', 'Weekend_Yes']

Data Types:
Temperature            float64
Humidity               float64
SquareFootage          float64
Occupancy                int64
RenewableEnergy        float64
Hour                     int64
Month                    int64
HVAC_Status              int64
Lighting_Status          int64
HVACUsage_On             int64
LightingUsage_On         int64
DayOfWeek_Monday         int64
DayOfWeek_Saturday       int64
DayOfWeek_Sunday         int64
DayOfWeek_Thursday       int64
DayOfWeek_Tuesday        int64
DayOfWeek_Wednesday      int64
Holiday_Yes              int64
Weekend_Yes              int64
dtype: object

Mi

In [11]:
# ============================================================
# C4 — Step 7: Train/Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_energy_encoded,
    y_energy,
    test_size=0.20,
    random_state=42
)

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("\nTraining Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Features Shape: (800, 19)
Testing Features Shape: (200, 19)

Training Target Shape: (800,)
Testing Target Shape: (200,)

Training Samples: 800
Testing Samples: 200


In [12]:
# ============================================================
# C4 — Step 8: Train Primary Energy Prediction Model
# ============================================================

start_time = time.time()

primary_model = LinearRegression()

primary_model.fit(X_train, y_train)

primary_train_time = time.time() - start_time

# Predictions
y_train_pred_primary = primary_model.predict(X_train)
y_test_pred_primary = primary_model.predict(X_test)

print("Primary Energy Model: Linear Regression")
print("Training Time:", round(primary_train_time, 4), "seconds")

print("\nTraining Predictions:", len(y_train_pred_primary))
print("Testing Predictions:", len(y_test_pred_primary))

Primary Energy Model: Linear Regression
Training Time: 0.0576 seconds

Training Predictions: 800
Testing Predictions: 200


In [13]:
# ============================================================
# C4 — Step 9: Evaluate Primary Energy Model
# ============================================================

def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)

    # Symmetric Mean Absolute Percentage Error
    smape = np.mean(
        2 * np.abs(y_pred - y_true) /
        (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    ) * 100

    # Normalized RMSE using target range
    nrmse = rmse / (np.max(y_true) - np.min(y_true)) * 100

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "sMAPE (%)": smape,
        "NRMSE (%)": nrmse,
        "Median AE": medae,
        "Explained Variance": evs
    }


primary_metrics = calculate_metrics(
    y_test,
    y_test_pred_primary
)

primary_metrics_df = pd.DataFrame(
    [primary_metrics],
    index=["C4 Primary Linear Regression"]
)

print(primary_metrics_df.round(4))

                                 MAE      MSE    RMSE      R2  sMAPE (%)  \
C4 Primary Linear Regression  4.0909  26.6679  5.1641  0.5929     5.3636   

                              NRMSE (%)  Median AE  Explained Variance  
C4 Primary Linear Regression    12.6992     3.4126               0.593  


In [15]:
# ============================================================
# C4 — Step 10A: Load Feature-Engineered Weather Dataset
# ============================================================

weather_fe_path = "../../Feature_engineered_datasets/weather_feature_engineered.csv"

weather_df = pd.read_csv(weather_fe_path)

print("Feature-Engineered Weather Dataset Shape:", weather_df.shape)

print("\nWeather Columns:")
print(weather_df.columns.tolist())

Feature-Engineered Weather Dataset Shape: (96429, 18)

Weather Columns:
['Formatted Date', 'Summary', 'Precip Type', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Pressure (millibars)', 'Daily Summary', 'Year', 'Month', 'Hour', 'DayOfWeek', 'Season', 'Temperature_Difference', 'Wind_Chill_Indicator']


In [16]:
# ============================================================
# C4 — Step 10: Select Weather Context Features
# ============================================================

weather_context_features = [
    "Temperature (C)",
    "Apparent Temperature (C)",
    "Humidity",
    "Wind Speed (km/h)",
    "Visibility (km)",
    "Pressure (millibars)"
]

weather_context = weather_df[
    ["Month", "Hour"] + weather_context_features
].copy()

print("Weather Context Shape:", weather_context.shape)

print("\nWeather Context Columns:")
print(weather_context.columns.tolist())

print("\nMissing Values:")
print(weather_context.isnull().sum())

Weather Context Shape: (96429, 8)

Weather Context Columns:
['Month', 'Hour', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)']

Missing Values:
Month                       0
Hour                        0
Temperature (C)             0
Apparent Temperature (C)    0
Humidity                    0
Wind Speed (km/h)           0
Visibility (km)             0
Pressure (millibars)        0
dtype: int64


In [17]:
# ============================================================
# C4 — Step 11: Aggregate Weather Context by Month + Hour
# ============================================================

weather_context_avg = (
    weather_context
    .groupby(["Month", "Hour"])[weather_context_features]
    .mean()
    .reset_index()
)

print("Aggregated Weather Context Shape:", weather_context_avg.shape)

print("\nFirst 10 Weather Context Groups:")
print(weather_context_avg.head(10))

print("\nNumber of Month-Hour Groups:",
      len(weather_context_avg))

Aggregated Weather Context Shape: (288, 8)

First 10 Weather Context Groups:
   Month  Hour  Temperature (C)  Apparent Temperature (C)  Humidity  \
0      1     0        -0.291105                 -2.963718  0.883607   
1      1     1        -0.410427                 -3.252786  0.881056   
2      1     2        -0.566846                 -3.283399  0.883959   
3      1     3        -0.614614                 -3.371717  0.889795   
4      1     4        -0.772988                 -3.598110  0.886041   
5      1     5        -0.790893                 -3.610101  0.887947   
6      1     6        -0.761518                 -3.637341  0.890528   
7      1     7        -0.457592                 -3.307739  0.884839   
8      1     8         0.391544                 -2.569762  0.864516   
9      1     9         1.311274                 -1.705279  0.841173   

   Wind Speed (km/h)  Visibility (km)  Pressure (millibars)  
0          10.346729         8.064353           1019.761290  
1          10.985

In [18]:
# ============================================================
# C4 — Step 12: Merge Energy Data with Weather Context
# ============================================================

energy_with_weather = energy_df.merge(
    weather_context_avg,
    on=["Month", "Hour"],
    how="left",
    suffixes=("", "_Weather")
)

print("Merged Dataset Shape:", energy_with_weather.shape)

print("\nMerged Columns:")
print(energy_with_weather.columns.tolist())

print("\nMissing Values After Merge:")
print(energy_with_weather[weather_context_features].isnull().sum())

print("\nFirst 5 Rows:")
print(
    energy_with_weather[
        ["Month", "Hour"] + weather_context_features
    ].head()
)

Merged Dataset Shape: (1000, 25)

Merged Columns:
['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status', 'Energy_Per_Occupant', 'Renewable_Energy_Ratio', 'Energy_Per_SquareFoot', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity_Weather', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)']

Missing Values After Merge:
Temperature (C)             0
Apparent Temperature (C)    0
Humidity                    0
Wind Speed (km/h)           0
Visibility (km)             0
Pressure (millibars)        0
dtype: int64

First 5 Rows:
   Month  Hour  Temperature (C)  Apparent Temperature (C)   Humidity  \
0      1     0        -0.291105                 -2.963718  43.431581   
1      1     1        -0.410427                 -3.252786  54.225919   
2      1     2        -0.566846                 -3.283399  58.907

In [19]:
# ============================================================
# C4 — Step 13: Prepare Weather Context Features
# ============================================================

weather_features_c4 = [
    "Temperature (C)",
    "Apparent Temperature (C)",
    "Humidity_Weather",
    "Wind Speed (km/h)",
    "Visibility (km)",
    "Pressure (millibars)"
]

X_weather_context = energy_with_weather[
    weather_features_c4
].copy()

print("Weather Context Feature Shape:", X_weather_context.shape)

print("\nWeather Context Features:")
print(X_weather_context.columns.tolist())

print("\nMissing Values:")
print(X_weather_context.isnull().sum())

Weather Context Feature Shape: (1000, 6)

Weather Context Features:
['Temperature (C)', 'Apparent Temperature (C)', 'Humidity_Weather', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)']

Missing Values:
Temperature (C)             0
Apparent Temperature (C)    0
Humidity_Weather            0
Wind Speed (km/h)           0
Visibility (km)             0
Pressure (millibars)        0
dtype: int64


In [20]:
# ============================================================
# C4 — Step 14: Align Weather Context with Energy Train/Test
# ============================================================

X_weather_train = X_weather_context.loc[X_train.index].copy()
X_weather_test = X_weather_context.loc[X_test.index].copy()

print("Weather Context Training Shape:", X_weather_train.shape)
print("Weather Context Testing Shape:", X_weather_test.shape)

print("\nEnergy Training Shape:", X_train.shape)
print("Energy Testing Shape:", X_test.shape)

print("\nIndex Alignment Check:")
print("Training indices match:",
      X_weather_train.index.equals(X_train.index))

print("Testing indices match:",
      X_weather_test.index.equals(X_test.index))

Weather Context Training Shape: (800, 6)
Weather Context Testing Shape: (200, 6)

Energy Training Shape: (800, 19)
Energy Testing Shape: (200, 19)

Index Alignment Check:
Training indices match: True
Testing indices match: True


In [21]:
# ============================================================
# C4 — Step 15: Generate Out-of-Fold Predictions
# ============================================================

from sklearn.model_selection import KFold, cross_val_predict

# 5-fold cross-validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

start_time = time.time()

oof_predictions = cross_val_predict(
    LinearRegression(),
    X_train,
    y_train,
    cv=kf,
    method="predict"
)

oof_time = time.time() - start_time

print("OOF Predictions Generated Successfully!")
print("Number of OOF Predictions:", len(oof_predictions))
print("OOF Generation Time:", round(oof_time, 4), "seconds")

print("\nFirst 10 OOF Predictions:")
print(oof_predictions[:10])

OOF Predictions Generated Successfully!
Number of OOF Predictions: 800
OOF Generation Time: 0.0318 seconds

First 10 OOF Predictions:
[70.96506705 68.54719263 72.75636549 77.38019256 81.21621636 79.06305442
 73.24379108 73.68455588 66.65000918 71.34423474]


In [22]:
# ============================================================
# C4 — Step 16: Create Residual Target
# ============================================================

# Residual = Actual Energy Consumption - OOF Primary Prediction
residual_target = y_train - oof_predictions

print("Residual Target Created Successfully!")

print("\nResidual Statistics:")
print(residual_target.describe())

print("\nFirst 10 Residual Values:")
print(residual_target.head(10).values)

print("\nMean Residual:", round(residual_target.mean(), 6))
print("Std Residual:", round(residual_target.std(), 6))

Residual Target Created Successfully!

Residual Statistics:
count    800.000000
mean      -0.022827
std        5.114088
min      -13.773392
25%       -3.593865
50%        0.067421
75%        3.557709
max       15.384093
Name: EnergyConsumption, dtype: float64

First 10 Residual Values:
[ 3.1952705  -3.7550209  -3.39074905  0.5624481  -1.16033279  0.36724649
 12.18388627 -7.12206695 -3.06787043  4.63403088]

Mean Residual: -0.022827
Std Residual: 5.114088


In [23]:
# ============================================================
# C4 — Step 17: Train Weather Residual Model
# ============================================================

start_time = time.time()

weather_residual_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

weather_residual_model.fit(
    X_weather_train,
    residual_target
)

weather_residual_train_time = time.time() - start_time

print("Weather Residual Model Trained Successfully!")
print("Model: Random Forest Regressor")
print("Number of Trees: 100")
print("Training Time:", round(weather_residual_train_time, 4), "seconds")

Weather Residual Model Trained Successfully!
Model: Random Forest Regressor
Number of Trees: 100
Training Time: 0.179 seconds


In [24]:
# ============================================================
# C4 — Step 18: Generate Final C4 Hybrid Predictions
# ============================================================

# Predict residual correction using Weather context
y_test_pred_residual = weather_residual_model.predict(
    X_weather_test
)

# Final C4 prediction
y_test_pred_c4 = (
    y_test_pred_primary
    + y_test_pred_residual
)

print("C4 Hybrid Predictions Generated Successfully!")

print("\nFirst 10 Predictions:")
comparison = pd.DataFrame({
    "Actual": y_test.values[:10],
    "Primary_LR": y_test_pred_primary[:10],
    "Weather_Correction": y_test_pred_residual[:10],
    "C4_Hybrid": y_test_pred_c4[:10]
})

print(comparison)

C4 Hybrid Predictions Generated Successfully!

First 10 Predictions:
      Actual  Primary_LR  Weather_Correction  C4_Hybrid
0  86.920611   83.505426            0.672502  84.177929
1  88.351606   81.246310            0.672502  81.918813
2  79.431363   75.821840            2.279505  78.101346
3  90.009188   88.240500           -1.270820  86.969680
4  83.891100   75.345495           -0.429023  74.916472
5  87.549041   80.923272           -1.560595  79.362678
6  79.697237   78.711368            0.068860  78.780228
7  80.914057   72.929673           -0.607118  72.322555
8  85.133856   77.779346            0.235705  78.015051
9  71.017140   67.828597            0.109036  67.937633


In [25]:
# ============================================================
# C4 — Step 19: Evaluate Final Hybrid Model
# ============================================================

c4_metrics = calculate_metrics(
    y_test,
    y_test_pred_c4
)

print("C4 Hybrid Model Results:")
print("=" * 50)

for metric, value in c4_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nPrimary Linear Regression Results:")
print("=" * 50)

for metric, value in calculate_metrics(
    y_test,
    y_test_pred_primary
).items():
    print(f"{metric}: {value:.4f}")

C4 Hybrid Model Results:
MAE: 4.2937
MSE: 28.3778
RMSE: 5.3271
R2: 0.5667
sMAPE (%): 5.6259
NRMSE (%): 13.1000
Median AE: 3.5520
Explained Variance: 0.5668

Primary Linear Regression Results:
MAE: 4.0909
MSE: 26.6679
RMSE: 5.1641
R2: 0.5929
sMAPE (%): 5.3636
NRMSE (%): 12.6992
Median AE: 3.4126
Explained Variance: 0.5930


In [26]:
# ============================================================
# C4 — Step 20: Create Energy–Weather Context Features
# ============================================================

c4_data = energy_with_weather.copy()

# Context interaction features
c4_data["Energy_Weather_Temp_Diff"] = (
    c4_data["Temperature"] -
    c4_data["Temperature (C)"]
)

c4_data["Energy_Weather_Humidity_Diff"] = (
    c4_data["Humidity"] -
    c4_data["Humidity_Weather"]
)

c4_data["Temperature_Humidity_Interaction"] = (
    c4_data["Temperature"] *
    c4_data["Humidity_Weather"]
)

c4_data["Occupancy_Temperature_Interaction"] = (
    c4_data["Occupancy"] *
    c4_data["Temperature (C)"]
)

print("C4 Context Features Created!")
print("\nNew columns:")

for col in [
    "Energy_Weather_Temp_Diff",
    "Energy_Weather_Humidity_Diff",
    "Temperature_Humidity_Interaction",
    "Occupancy_Temperature_Interaction"
]:
    print("-", col)

print("\nShape:", c4_data.shape)

C4 Context Features Created!

New columns:
- Energy_Weather_Temp_Diff
- Energy_Weather_Humidity_Diff
- Temperature_Humidity_Interaction
- Occupancy_Temperature_Interaction

Shape: (1000, 29)


In [27]:
# ============================================================
# C4 — Step 21: Expanded Weather Context Branch
# ============================================================

weather_context_features_c4_v2 = [
    "Temperature (C)",
    "Apparent Temperature (C)",
    "Humidity_Weather",
    "Wind Speed (km/h)",
    "Visibility (km)",
    "Pressure (millibars)",
    "Energy_Weather_Temp_Diff",
    "Energy_Weather_Humidity_Diff",
    "Temperature_Humidity_Interaction",
    "Occupancy_Temperature_Interaction"
]

X_weather_context_v2 = c4_data[
    weather_context_features_c4_v2
].copy()

X_weather_train_v2 = X_weather_context_v2.loc[X_train.index].copy()
X_weather_test_v2 = X_weather_context_v2.loc[X_test.index].copy()

print("Expanded Weather Context Training Shape:",
      X_weather_train_v2.shape)

print("Expanded Weather Context Testing Shape:",
      X_weather_test_v2.shape)

print("\nMissing Values:",
      X_weather_context_v2.isna().sum().sum())

print("\nIndex Alignment:")
print(
    "Train:",
    X_weather_train_v2.index.equals(X_train.index)
)

print(
    "Test:",
    X_weather_test_v2.index.equals(X_test.index)
)

Expanded Weather Context Training Shape: (800, 10)
Expanded Weather Context Testing Shape: (200, 10)

Missing Values: 0

Index Alignment:
Train: True
Test: True


In [28]:
# ============================================================
# C4 — Step 22: Train Expanded Weather Residual Model
# ============================================================

start_time = time.time()

weather_residual_model_v2 = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

weather_residual_model_v2.fit(
    X_weather_train_v2,
    residual_target
)

weather_residual_train_time_v2 = time.time() - start_time

print("Expanded Weather Residual Model Trained Successfully!")

print("Model: Random Forest Regressor")
print("Number of Trees: 100")
print(
    "Training Time:",
    round(weather_residual_train_time_v2, 4),
    "seconds"
)

Expanded Weather Residual Model Trained Successfully!
Model: Random Forest Regressor
Number of Trees: 100
Training Time: 0.2328 seconds


In [29]:
# ============================================================
# C4 — Step 23: Generate Improved C4 Predictions
# ============================================================

# Predict weather-based residual correction
y_test_pred_residual_v2 = weather_residual_model_v2.predict(
    X_weather_test_v2
)

# Final improved C4 prediction
y_test_pred_c4_v2 = (
    y_test_pred_primary
    + y_test_pred_residual_v2
)

print("Improved C4 Predictions Generated Successfully!")

comparison_v2 = pd.DataFrame({
    "Actual": y_test.values[:10],
    "Primary_LR": y_test_pred_primary[:10],
    "Weather_Correction": y_test_pred_residual_v2[:10],
    "C4_Hybrid_v2": y_test_pred_c4_v2[:10]
})

print(comparison_v2)

Improved C4 Predictions Generated Successfully!
      Actual  Primary_LR  Weather_Correction  C4_Hybrid_v2
0  86.920611   83.505426           -0.642888     82.862538
1  88.351606   81.246310            2.887229     84.133539
2  79.431363   75.821840            1.304340     77.126180
3  90.009188   88.240500           -1.286333     86.954167
4  83.891100   75.345495            0.898037     76.243532
5  87.549041   80.923272           -2.266496     78.656777
6  79.697237   78.711368            0.051838     78.763206
7  80.914057   72.929673            0.119463     73.049137
8  85.133856   77.779346           -1.106705     76.672641
9  71.017140   67.828597            0.465387     68.293984


In [30]:
# ============================================================
# C4 — Step 24: Evaluate Improved C4
# ============================================================

c4_v2_metrics = calculate_metrics(
    y_test,
    y_test_pred_c4_v2
)

primary_metrics = calculate_metrics(
    y_test,
    y_test_pred_primary
)

print("C4 Hybrid v2 Results")
print("=" * 50)

for metric, value in c4_v2_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nPrimary Linear Regression")
print("=" * 50)

for metric, value in primary_metrics.items():
    print(f"{metric}: {value:.4f}")

# Improvement calculations
rmse_improvement = (
    (primary_metrics["RMSE"] - c4_v2_metrics["RMSE"])
    / primary_metrics["RMSE"]
) * 100

r2_change = (
    (c4_v2_metrics["R2"] - primary_metrics["R2"])
    / abs(primary_metrics["R2"])
) * 100

mae_improvement = (
    (primary_metrics["MAE"] - c4_v2_metrics["MAE"])
    / primary_metrics["MAE"]
) * 100

print("\nC4 v2 Comparison")
print("=" * 50)
print(f"RMSE Improvement: {rmse_improvement:.2f}%")
print(f"R² Change:        {r2_change:.2f}%")
print(f"MAE Improvement:  {mae_improvement:.2f}%")

C4 Hybrid v2 Results
MAE: 4.4360
MSE: 30.8172
RMSE: 5.5513
R2: 0.5295
sMAPE (%): 5.8208
NRMSE (%): 13.6514
Median AE: 4.0217
Explained Variance: 0.5298

Primary Linear Regression
MAE: 4.0909
MSE: 26.6679
RMSE: 5.1641
R2: 0.5929
sMAPE (%): 5.3636
NRMSE (%): 12.6992
Median AE: 3.4126
Explained Variance: 0.5930

C4 v2 Comparison
RMSE Improvement: -7.50%
R² Change:        -10.69%
MAE Improvement:  -8.44%
